# DeepSleep 2.0 - Model Training & Evaluation

Run the cells top-to-bottom. Two typical workflows:

* **Evaluate a shipped model** (`TRAIN_MODE = False`): loads the checkpoints in `models/<MODEL_NAME>/`,
  picks the epoch with the lowest validation loss and reports the test loss and gross AUROC/AUPRC.
  The cached `test_*.pickle` results are reused when present, so no dataset (and no GPU) is needed
  for this. Without the cache, the PhysioNet `training` folder must be present under `data/`.
* **Train a model** (`TRAIN_MODE = True`): requires the PhysioNet `training` folder under `data/`
  (see `data/README.md`) and, realistically, a CUDA GPU. Set `LOAD_CHECKPOINT = True` to resume.

## Main Switches

In [ ]:
'''
MODEL:
Select an existing model configuration (see the 'models' folder and paper) or
define a custom configuration in './models/<name>/hyperparameters.txt' (JSON)
'''
MODEL_NAME = 'model_2' # available: {'model_0'; 'model_1'; 'model_2'; 'model_3'}

'''
MODE:
Select between model training (True) or evaluation of existing checkpoints (False).
In evaluation mode the latest checkpoint is always loaded (LOAD_CHECKPOINT is ignored).
'''
TRAIN_MODE = True  # available options: {True; False}

'''
CHECKPOINT (training mode only):
Resume training from the latest checkpoint in the model folder? If False, training
starts from scratch and overwrites any existing checkpoints in that folder.
'''
LOAD_CHECKPOINT = False  # available options: {True; False}

## Modules

In [ ]:
import os
import random
import numpy as np
import pickle
import json
import csv
import time
import importlib
import matplotlib.pyplot as plt
from itertools import zip_longest
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
# Model-summary backend: prefer the maintained 'torchinfo', fall back to legacy 'torchsummary'.
try:
    from torchinfo import summary
    _SUMMARY_BACKEND = 'torchinfo'
except ImportError:
    from torchsummary import summary
    _SUMMARY_BACKEND = 'torchsummary'
from tqdm import tqdm
from losses import CustomBCELoss, CustomBCEWithLogitsLoss
from score2018 import Challenge2018Score
from utils import preprocess, folders_to_records_txt, latest_checkpoint
from dataset import CustomDataset
from transforms import ToTensor, build_train_transforms

# --- Mixed-precision (AMP) compatibility shim ---
# Uses the modern `torch.amp` API on PyTorch >= 2.4 and transparently falls back to
# the older `torch.cuda.amp` API on legacy installs (e.g. the paper's torch 1.9).
def amp_autocast(enabled=True):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        try:
            return torch.amp.autocast('cuda', enabled=enabled)
        except TypeError:
            pass
    return torch.cuda.amp.autocast(enabled=enabled)

def amp_grad_scaler(enabled=True):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        try:
            return torch.amp.GradScaler('cuda', enabled=enabled)
        except TypeError:
            pass
    return torch.cuda.amp.GradScaler(enabled=enabled)

print('GPU is', '' if torch.cuda.is_available() else 'not', 'available.')

## Paths

In [ ]:
ROOT_PATH        = './'
DATA_PATH        = os.path.join(ROOT_PATH, 'data')
TRAIN_DATA_PATH  = os.path.join(DATA_PATH, 'training')
TEST_DATA_PATH   = os.path.join(DATA_PATH, 'test')
MODEL_PATH       = os.path.join(ROOT_PATH, 'models', MODEL_NAME)

## Hyperparameters and Reproducibility

In [ ]:
assert os.path.isfile(os.path.join(MODEL_PATH, 'hyperparameters.txt')), \
"File hyperparameters.txt does not exist in: " + MODEL_PATH + os.path.sep

# load the model configuration and hyperparameters from the JSON file
with open(os.path.join(MODEL_PATH, 'hyperparameters.txt')) as f:
    hyperparameters = json.load(f)
print('\033[1m',"Hyperparameters specific to the selected model:",'\033[0m', \
      *hyperparameters.items(), sep='\n')

ARCHITECTURE_NAME = hyperparameters['ARCHITECTURE_NAME']
SEED              = hyperparameters['SEED']
MAX_NUM_EPOCHS    = hyperparameters['MAX_NUM_EPOCHS']
CHANNELS          = hyperparameters['CHANNELS']
LEARNING_RATE     = hyperparameters['LEARNING_RATE']
DECAY_RATE        = hyperparameters['DECAY_RATE']
STOP_STRIP        = hyperparameters['STOP_STRIP']
DEVICE            = hyperparameters['DEVICE']
BATCH_SIZE        = hyperparameters['BATCH_SIZE']
NUM_WORKERS       = hyperparameters['NUM_WORKERS']
PIN_MEMORY        = hyperparameters['PIN_MEMORY']
LINEAR            = hyperparameters['LINEAR']
Z_NORM            = hyperparameters['Z_NORM']
TRANSFORMS        = hyperparameters['TRANSFORMS']

# h5py channel selection requires a strictly increasing list of valid channel indices
assert CHANNELS and CHANNELS == sorted(set(CHANNELS)) and set(CHANNELS) <= set(range(13)), \
"CHANNELS must be a strictly increasing list of channel indices in 0..12"

# fall back to the CPU when no GPU is available
if not torch.cuda.is_available():
    if str(DEVICE).startswith('cuda'):
        print("CUDA is not available, using the CPU instead of", DEVICE,
              "(training on the CPU is extremely slow)." if TRAIN_MODE else "")
    DEVICE = "cpu"
    PIN_MEMORY = False

# to enable analysis/inference on CPU-only devices
if not TRAIN_MODE:
    BATCH_SIZE = 2 if torch.cuda.is_available() else 1
    NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# set seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Data Availability & Preprocessing

In [ ]:
'''
Only the PhysioNet 'training' folder is used: it is the only one with arousal labels.
The PhysioNet 'test' folder is unlabeled and therefore cannot be used for evaluation.
'''
TRAIN_DATA_AVAILABLE = os.path.isdir(TRAIN_DATA_PATH) and \
    any(not d.startswith('.') and os.path.isdir(os.path.join(TRAIN_DATA_PATH, d))
        for d in os.listdir(TRAIN_DATA_PATH))

if os.path.isdir(TEST_DATA_PATH):
    print("Note: the PhysioNet 'test' folder is unlabeled and will be ignored.")

if TRAIN_DATA_AVAILABLE:
    '''
    Identify all records stored in '/data/*' (incl. subfolders) and save
    their respective paths in 'RECORDS.txt'.
    '''
    folders_to_records_txt(DATA_PATH)

    '''
    Uniform all recording/arousal signals to have the same 8-million length
    (2^23 = 8,388,608) by padding zeros and centering the recording/arousal
    signals. Perform Z-score normalization, if applicable.
    '''
    preprocess(TRAIN_DATA_PATH, Z_NORM)
elif TRAIN_MODE:
    raise FileNotFoundError("Training requires the PhysioNet 'training' folder in " +
                            TRAIN_DATA_PATH + " (see data/README.md).")
else:
    print("PhysioNet 'training' folder not found in", TRAIN_DATA_PATH,
          "- only cached test results can be shown.")

## Custom Dataset & Transforms

`CustomDataset` (in [`dataset.py`](dataset.py)) reads the cached `.h5py` records. The training-only
augmentations (`MagScale`, `MagScaleRandCh`, `RandShuffle`, `AddRandGaussian2All`,
`InjectRandGaussian`, `TimeScale`) live in [`transforms.py`](transforms.py) and are selected by the
`TRANSFORMS` hyperparameter. Both are kept in modules so that DataLoader worker processes can load
them on macOS/Windows.

## Dataset Split

In [ ]:
'''
The PhysioNet training dataset is split in training (60%), validation (15%),
and test (25%) sets. If the model folder already holds a 'records.csv' file
(as all shipped models do), that split is reused, so evaluation and retraining
use exactly the same records as the paper. Delete 'records.csv' to draw a new split.
'''
records_csv_path = os.path.join(MODEL_PATH, 'records.csv')

if os.path.isfile(records_csv_path):
    with open(records_csv_path, newline = '') as f:
        columns = list(zip(*csv.reader(f)))
    splits = {col[0]: [r for r in col[1:] if r] for col in columns}
    train_records = splits['train_records']
    val_records   = splits['val_records']
    test_records  = splits['test_records']
    print("Reusing the split stored in", records_csv_path)

else:
    assert os.path.isfile(os.path.join(TRAIN_DATA_PATH, 'RECORDS.txt')), \
    "File RECORDS.txt does not exist in: " + TRAIN_DATA_PATH + os.path.sep

    with open(os.path.join(TRAIN_DATA_PATH, 'RECORDS.txt'), 'r') as f:
        records_in_train_folder = sorted(f.read().splitlines())

    train_tot   = len(records_in_train_folder)
    train_split = int(0.6*train_tot)
    val_split   = int(0.15*train_tot)
    test_split  = train_tot - train_split - val_split

    train_records, tmp = train_test_split(records_in_train_folder, \
                                          test_size = val_split + test_split, \
                                          random_state = SEED)
    val_records, test_records = train_test_split(tmp, test_size = test_split, \
                                                 random_state = SEED)

    # record the splits
    if TRAIN_MODE:
        with open(records_csv_path, "w", newline = '') as f:
            writer = csv.writer(f)
            writer.writerow(['train_records', 'val_records', 'test_records'])
            for values in zip_longest(*[train_records, val_records, test_records]):
                writer.writerow(values)

n_tot = len(train_records) + len(val_records) + len(test_records)
print("Split: training {} ({:.0%}), validation {} ({:.0%}), test {} ({:.0%}) records.".format(
      len(train_records), len(train_records)/n_tot, len(val_records), len(val_records)/n_tot,
      len(test_records), len(test_records)/n_tot))

# make sure every record of the split has been downloaded and preprocessed
if TRAIN_DATA_AVAILABLE:
    file_add = '_norm' if Z_NORM else ''
    missing = [r for r in train_records + val_records + test_records
               if not (os.path.isfile(os.path.join(TRAIN_DATA_PATH, r + file_add + '.h5py')) and
                       os.path.isfile(os.path.join(TRAIN_DATA_PATH, r + '-arousal.h5py')))]
    assert not missing, "{} of the {} records in the split are missing in {} (e.g. {}). " \
        "Is the download complete?".format(len(missing), n_tot, TRAIN_DATA_PATH, missing[:3])

## Dataloader

In [ ]:
train_transforms = build_train_transforms(TRANSFORMS, CHANNELS, Z_NORM)
val_test_transforms = ToTensor()

train_dataset = CustomDataset(TRAIN_DATA_PATH, train_records, CHANNELS, \
                              Z_NORM, train_transforms)
val_dataset   = CustomDataset(TRAIN_DATA_PATH, val_records, CHANNELS, \
                              Z_NORM, val_test_transforms)
test_dataset  = CustomDataset(TRAIN_DATA_PATH, test_records, CHANNELS, \
                              Z_NORM, val_test_transforms)

print("Length of the train dataset is:", len(train_dataset))
print("Length of the val dataset is:", len(val_dataset))
print("Length of the test dataset is:", len(test_dataset))

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle = True, \
                          num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY)
val_loader   = DataLoader(val_dataset, BATCH_SIZE, shuffle = False, \
                          num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY)
test_loader  = DataLoader(test_dataset, BATCH_SIZE, shuffle = False, \
                          num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY)

print("Total number of train batches:", len(train_loader))
print("Total number of val batches:", len(val_loader))
print("Total number of test batches:", len(test_loader))

## Evaluation Function

In [ ]:
def eval_fn(model, loader, DEVICE, comp_score = False):
    
    # prepare model for evaluation (disable batch normalization)
    model.eval()
    
    # use the official PhysioNet scoring function
    scores = Challenge2018Score() if comp_score else None
    
    # define loss function
    if comp_score:
        loss_fn = CustomBCELoss().to(DEVICE)
    else:
        loss_fn = CustomBCEWithLogitsLoss().to(DEVICE)
    
    # compute loss and, if applicable, score
    with torch.no_grad():
        loss_epoch_sum = 0
        for x, y, idx in loader:          
            x = x.to(device = DEVICE)
            y = y.to(device = DEVICE)

            with amp_autocast(enabled = torch.cuda.is_available() and not comp_score):

                y_hat = model(x, comp_score)
                loss = loss_fn(y_hat, y)
            
                # compute AUROC/AUPRC score for each record in batch
                if comp_score:                
                    for i, single_idx in enumerate(idx):
                        record = loader.dataset.get_record_name(single_idx)
                        scores.score_record(y[i].view(-1).to('cpu'), \
                                            y_hat[i].view(-1).to('cpu'), record)
                        auroc = scores.record_auroc(record)
                        auprc = scores.record_auprc(record)               
                        print('%-11s  AUROC: %8.6f,  AUPRC: %8.6f' % \
                              (record, auroc, auprc))

            loss_epoch_sum += float(loss.item())                 

    return loss_epoch_sum/len(loader), scores

## Training Function

In [ ]:
def train_fn(model, loader, optimizer, loss_fn, scaler, DEVICE):

    # prepare model for training (enable batch normalization updates)
    model.train()

    loss_epoch_sum = 0

    with tqdm(total = len(loader)) as pbar:
        start_time = time.time()

        for batch_idx, (x, y, _) in enumerate(loader):

            """ Step 0. move the tensors to the right device """

            x = x.to(device = DEVICE)
            y = y.to(device = DEVICE)

            # time spent loading the batch and moving it to the device
            prepare_time = time.time() - start_time

            """ Step 1. clear gradients """
            optimizer.zero_grad(set_to_none = True)

            with amp_autocast(enabled = torch.cuda.is_available()):

                """ Step 2. Forward pass """
                y_hat = model(x)

                """ Step 3. Loss calculation """
                loss = loss_fn(y_hat, y)

            """ Step 4. Backward pass """
            scaler.scale(loss).backward()

            """ Step 5. Optimization (parameter update) """
            scaler.step(optimizer)
            scaler.update()

            """ Step 6. Timing and logging """
            loss_epoch_sum += float(loss.item())

            process_time = time.time() - start_time - prepare_time
            compute_efficiency = process_time / (process_time + prepare_time)

            pbar.update(1)
            pbar.set_postfix({'Running loss' : loss_epoch_sum/(batch_idx + 1), \
                              'Compute efficiency': compute_efficiency})
            start_time = time.time()

    return loss_epoch_sum/len(loader)

## Training Loop

In [ ]:
'''
Create (and load) model and define hyperparameters
'''

DeepSleepNet = getattr(importlib.import_module('architectures.' + ARCHITECTURE_NAME), 'DeepSleepNet')

model     = DeepSleepNet(in_channels = len(CHANNELS), linear = LINEAR).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay = DECAY_RATE)
loss_fn   = CustomBCEWithLogitsLoss().to(DEVICE)
scaler    = amp_grad_scaler(enabled = torch.cuda.is_available())

epoch0             = 0
train_loss_history = []
val_loss_history   = []
epochs_trained     = 0  # number of epochs trained in this session
last_checkpoint_path = latest_checkpoint(MODEL_PATH)

if LOAD_CHECKPOINT or not TRAIN_MODE:
    if last_checkpoint_path is None:
        if TRAIN_MODE:
            print("No checkpoint found in", MODEL_PATH, "- training starts from scratch.")
        else:
            raise FileNotFoundError("Evaluation mode (TRAIN_MODE = False) needs trained "
                                    "checkpoints, but none was found in " + MODEL_PATH)
    else:
        checkpoint = torch.load(last_checkpoint_path, \
                                map_location = torch.device(DEVICE), weights_only = False)
        model.load_state_dict(checkpoint['model_state_dict'])
        if TRAIN_MODE:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch0 = checkpoint['epoch'] + 1
        train_loss_history = checkpoint['train_loss_history']
        val_loss_history = checkpoint['val_loss_history']
        print('Model saved in', os.path.basename(last_checkpoint_path), 'loaded.')
elif last_checkpoint_path is not None:
    print("Warning: training from scratch; existing checkpoints in", MODEL_PATH,
          "will be overwritten.")

'''
Start the training procedure
'''
if TRAIN_MODE:
    for epoch in range(epoch0, MAX_NUM_EPOCHS):
        np.random.seed(SEED + epoch)

        # evaluate early stop criterion
        if len(val_loss_history) > STOP_STRIP:
            if val_loss_history[-(STOP_STRIP + 1)] < min(val_loss_history[-STOP_STRIP::]) \
            or np.isnan([val_loss_history, train_loss_history]).any():
                print("Early stop has been triggered.")
                break

        # run one epoch training
        train_loss_epoch = train_fn(model, train_loader, optimizer, loss_fn, scaler, DEVICE)

        # check validation loss
        val_loss_epoch, _ = eval_fn(model, val_loader, DEVICE)

        # loss logging
        train_loss_history.append(train_loss_epoch)
        val_loss_history.append(val_loss_epoch)
        epochs_trained += 1

        # loss printing
        print("\nEpoch: {} \t Mean training loss {:.6f}".format(epoch, train_loss_epoch))
        print("Epoch: {} \t Mean validation loss {:.6f}".format(epoch, val_loss_epoch))

        # save current epoch model checkpoint
        checkpoint = {
            "epoch"                : epoch,
            "model_state_dict"     : model.state_dict(),
            "optimizer_state_dict" : optimizer.state_dict(),
            "train_loss_history"   : train_loss_history,
            "val_loss_history"     : val_loss_history,
        }
        torch.save(checkpoint, os.path.join(MODEL_PATH, \
                                            'my_checkpoint_'+str(epoch)+'.pth.tar'))
        print(">Checkpoint {} saved<".format(epoch))

## Training & Validation Loss Histories

In [ ]:
plt.plot(range(len(train_loss_history)), train_loss_history, '-rx', \
         label = "Training loss")
plt.plot(range(len(val_loss_history)), val_loss_history, '-bx', \
         label = "Validation loss")
plt.xlabel("Epoch (#)", fontdict = None, labelpad = None)
plt.ylabel("Cross Entropy Loss", fontdict = None, labelpad = None)
plt.legend(loc = 'upper right', borderaxespad = 0.7, shadow = True)
plt.grid(linestyle = '--', linewidth = 0.1)

## Identify the Smallest Cross-Validation Loss Model

In [ ]:
assert val_loss_history, "No validation loss history available (no checkpoint loaded or trained)."

best_model_idx = val_loss_history.index(min(val_loss_history))
best_model_path = os.path.join(MODEL_PATH, 'my_checkpoint_' + str(best_model_idx) + '.pth.tar')
checkpoint = torch.load(best_model_path, map_location = torch.device(DEVICE), weights_only = False)
model.load_state_dict(checkpoint['model_state_dict'])
print("Best model has been loaded from my_checkpoint_" + str(best_model_idx) + ".pth.tar")

## Compute Test Loss and Gross AUROC/AUPRC

In [ ]:
test_loss_path = os.path.join(MODEL_PATH, 'test_loss_' + str(best_model_idx) + '.pickle')
test_score_path = os.path.join(MODEL_PATH, 'test_score_' + str(best_model_idx) + '.pickle')

# cached results are only trusted if no new training happened in this session
use_cache = epochs_trained == 0 and os.path.exists(test_loss_path) and os.path.exists(test_score_path)

if use_cache:
    print("Loading cached test results.")
    with open(test_loss_path, 'rb') as f1, open(test_score_path, 'rb') as f2:
        test_loss = pickle.load(f1)
        test_score = pickle.load(f2)
else:
    if not TRAIN_DATA_AVAILABLE:
        raise FileNotFoundError("No cached test results for epoch " + str(best_model_idx) +
                                "; computing them requires the PhysioNet 'training' folder in " +
                                TRAIN_DATA_PATH)
    test_loss, test_score = eval_fn(model, test_loader, DEVICE, comp_score = True)
    with open(test_loss_path, 'wb') as f1, open(test_score_path, 'wb') as f2:
        pickle.dump(test_loss, f1)
        pickle.dump(test_score, f2)

print("Test results based on {:d} test cases.".format(len(test_score._record_auc)))
print("Cross-entropy loss = {:.6f}".format(test_loss))
print('Gross AUROC: %8.6f' % test_score.gross_auroc())
print('Gross AUPRC: %8.6f' % test_score.gross_auprc())

## Model Summary

In [ ]:
# A full-length (2^23) input needs several GB of memory, so the CPU summary uses a short
# input; the layer structure and parameter count are the same for any length that is a
# multiple of 2^14 (the product of the pooling factors 4*8*16*32).
SUMMARY_LEN = 2**23 if str(DEVICE).startswith('cuda') else 2**14

if _SUMMARY_BACKEND == 'torchinfo':
    summary(model, input_size=(1, len(CHANNELS), SUMMARY_LEN), device=DEVICE)
else:
    summary(model, input_size=(len(CHANNELS), SUMMARY_LEN), device=str(DEVICE))